# step C (생성 기반) — 치환 후 실제로 준수 표기를 쓰는가

**대응 RQ:** RQ2 인과의 **직관적 재확인**. 선호 점수(stepC_kv-intervention)가 이미 67% 회복을 보였고,
여기서는 텍스트는 그대로 둔 채 **L25 KV를 준수 값으로 치환한 뒤 이름을 실제로 생성**해 표기를 분류한다.

**측정:** 그리디 생성으로 나온 첫 함수 이름의 표기(camel/snake). 치환 **전(baseline)** vs **후(intervened)** 준수율.
- **baseline:** 개입 없음 — 위반 선행이라 위반 표기로 쓸 것.
- **intervened + compliant:** L25를 같은 이름 준수판으로 치환 → 준수로 바뀌는가(주효과).
- **+ unrelated_camel:** 무관 준수형(형태 통제) / **+ unrelated_snake:** 무관 위반형(음성 통제).

설계: `docs/stepC/plan.md`.

> **주의:** 생성은 이진 결과라 선호 점수보다 노이즈가 크다(seed 20). 선호 점수 결과가 정밀한 본 측정, 이건 그림.
> **재개 가능:** 조건마다 저장. GPU 없으면 매우 느림.

In [ ]:
# 환경 설정
!pip install -q transformers accelerate torch matplotlib pandas
import random, numpy as np, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
SEED=0; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
# 클론·브랜치
import os
if not os.path.isdir('HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git
%cd HCLT_2026
!git fetch --quiet origin stepC/KV-intervention
!git checkout stepC/KV-intervention
!git pull --quiet origin stepC/KV-intervention
!pip install -e . -q
import sys; sys.path.insert(0,'src')

In [ ]:
# 조건 — donor 3종 × seed. 선호 점수 실험과 동일 조건, 측정만 생성 기반.
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Composition, InstructionForm, Notation, Intervention, InterventionKind)
MODEL = ModelSpec(name='Qwen/Qwen2.5-Coder-3B-Instruct', family='qwen', dtype='float16')
DONORS = ['compliant', 'unrelated_camel', 'unrelated_snake']
SEEDS = list(range(20)); LAYER = 25
def make(donor, s):
    return Condition(model=MODEL,
        preceding=PrecedingCode(n_compliant=0, n_functions=12, composition=Composition.POOL),
        instruction=Instruction(form=InstructionForm.POSITIVE, target_notation=Notation.CAMEL),
        intervention=Intervention(kind=InterventionKind.KEY_VALUE, layers=[LAYER], donor=donor),
        seed=s)
conditions = [make(d, s) for d in DONORS for s in SEEDS]
PREDICTION = ('baseline은 위반 표기. compliant/unrelated_camel 치환 후 준수 표기로 상승. '
              'unrelated_snake는 그대로 위반.')
print(len(conditions), '조건')

In [ ]:
# 실행 — mode='intervene_generate'. 중간중간 baseline vs intervened 준수율.
from collections import defaultdict
from harness import run, ResultRecord, save_result, result_path
from harness.results import load_result
from harness.model import load_model
STEP = 'stepC-gen'
handle = load_model(MODEL)
print('layers:', handle.num_layers)
base_acc=[]; int_acc=defaultdict(list)
new=skipped=0
for i,c in enumerate(conditions,1):
    p = result_path(c, step=STEP)
    if p.exists(): rec=load_result(p); skipped+=1
    else:
        out = run(c, handle=handle, mode='intervene_generate')
        save_result(ResultRecord(condition=out.condition, metrics=out.metrics, step=STEP, rq='RQ2', prediction=PREDICTION))
        rec=load_result(p); new+=1
    ex=rec.metrics.extra; d=rec.condition.intervention.donor
    int_acc[d].append(ex['intervened_compliant'])
    if d=='compliant': base_acc.append(ex['baseline_compliant'])
    if i%15==0 or i==len(conditions):
        print(f'[{i}/{len(conditions)}] 새 {new} 건너뜀 {skipped}')
        if base_acc: print(f'    baseline 준수율 {sum(base_acc)/len(base_acc):.2f}')
        for d in DONORS:
            if int_acc[d]: print(f'    치환후({d}) 준수율 {sum(int_acc[d])/len(int_acc[d]):.2f}')
print('완료')

In [ ]:
# 결과 로드
from harness import result_path
from harness.results import load_result
records = [load_result(result_path(c, step='stepC-gen')) for c in conditions]
print('로드:', len(records))

In [ ]:
# 요약 — baseline vs 치환후 준수율 + 생성 이름 예시
import pandas as pd, matplotlib.pyplot as plt
rows=[{'donor':r.condition.intervention.donor,
       'baseline':r.metrics.extra['baseline_compliant'],
       'intervened':r.metrics.extra['intervened_compliant'],
       'name_base':r.metrics.extra['name_baseline'],
       'name_int':r.metrics.extra['name_intervened']} for r in records]
df=pd.DataFrame(rows)
base_rate=df[df.donor=='compliant']['baseline'].mean()
print('baseline 준수율:', round(base_rate,3))
print('치환후 준수율(donor별):'); print(df.groupby('donor')['intervened'].mean().round(3))
print('\n생성 이름 예시 (donor=compliant, 앞 5):')
for _,r in df[df.donor=='compliant'].head(5).iterrows():
    print(f"  baseline={r['name_base']!r:<22} intervened={r['name_int']!r}")
# 플롯
labels=['baseline']+['int:'+d for d in ['compliant','unrelated_camel','unrelated_snake']]
vals=[base_rate]+[df[df.donor==d]['intervened'].mean() for d in ['compliant','unrelated_camel','unrelated_snake']]
plt.figure(figsize=(6.4,3.4))
plt.bar(range(4), vals, color=['#B0392B','#2E7D52','#2563C9','#C6771A'])
plt.xticks(range(4), labels, rotation=12, fontsize=8.5); plt.ylim(0,1.05); plt.ylabel('compliance rate')
plt.title('Generated-name compliance: baseline vs L25 KV substitution'); plt.tight_layout(); plt.show()

In [ ]:
# 결과 다운로드
import shutil
shutil.make_archive('stepC_gen_results','zip','results/stepC-gen')
try:
    from google.colab import files; files.download('stepC_gen_results.zip')
except Exception as e: print('Colab 아님:', e)